In [2]:
import pandas as pd
import plotly.express as px
import requests
import time
import re

In [3]:
dfs = {}
years = ['2021', '2022', '2023', '2024', '2025']

for year in years:
    dfs[year] = pd.read_csv(f'financial_data_{year}.csv')

/var/folders/9r/612l79zx42z643jnmxxshb1m0000gn/T/ipykernel_1197/448918627.py:5: DtypeWarning: Columns (0: expl2510) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[year] = pd.read_csv(f'financial_data_{year}.csv')
/var/folders/9r/612l79zx42z643jnmxxshb1m0000gn/T/ipykernel_1197/448918627.py:5: DtypeWarning: Columns (0: expl2300, 1: expl2900, 2: expl2100, 3: expl2500, 4: expl2310, 5: expl2510) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[year] = pd.read_csv(f'financial_data_{year}.csv')
/var/folders/9r/612l79zx42z643jnmxxshb1m0000gn/T/ipykernel_1197/448918627.py:5: DtypeWarning: Columns (0: expl2400, 1: expl2900, 2: expl2100, 3: expl2500) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[year] = pd.read_csv(f'financial_data_{year}.csv')
/var/folders/9r/612l79zx42z643jnmxxshb1m0000gn/T/ipykernel_1197/448918627.py:5: DtypeWarning: Columns (0: expl2300, 1: expl2900, 2: expl2500, 3: expl2310) hav

**Подготовка датафрейма df_top500**

In [4]:
revenue_all_years = pd.DataFrame()
for year in years:
    revenue_all_years[year] = dfs[year]['current2110']

total_revenue = revenue_all_years.sum(axis=1)

top500_id = total_revenue.nlargest(500).index

df_top500 = pd.DataFrame({
    'total_revenue': total_revenue[top500_id],
    'company_name': dfs['2021'].loc[top500_id, 'company_name'],
    'address': dfs['2021'].loc[top500_id, 'address']
})

df_top500 = df_top500.reset_index(drop = True)
df_top500

,total_revenue,company_name,address
0,78306425.0,"ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ""МОНТ...","643,628012,86,,Ханты-Мансийск г,,Чехова ул,70,,"
1,18809902.0,"Общество с ограниченной ответственностью ""НЕФА...","119313, Москва г, Ленинский пр-кт, д. № 95, эт..."
2,15432504.0,"ООО ""Стройснаб""","632640, Новосибирская обл, рп.Коченево, ул.Пуш..."
3,14566762.0,"Общество с ограниченной ответственностью ""КВ-С...","241050, г. Брянск, ул. Советская, д. 49, к. 1,..."
4,14177891.0,"Общество с ограниченной ответственностью ""Стро...","241020, Брянская обл, Брянск г, Уральская ул, ..."
...,...,...,...
495,3698827.0,"Общество с ограниченной ответственностью ""Жилп...","420012, Татарстан Респ, Казань г, Бутлерова ул..."
496,3698814.0,"ООО ""СлавянСтрой""","308000, Белгородская обл, Белгород г, Народный..."
497,3693783.0,"Общество с ограниченной ответственностью ""КОМП...","109428, Москва г, Рязанский пр-кт, д. № 10, ст..."
498,3691597.0,"Общество с ограниченной ответственностью ""Упра...","140002, Московская обл, Люберцы г, Кирова ул, ..."


**Поиск координат по адресам**

In [5]:
print('Введите свой API для Яндекс гео:')
API_KEY = input()

def get_coords_yandex(address):
    clean = address.replace(",,", ",").replace("  ", " ")
    clean = re.sub(r'^\d+,?\d*,?\d*,?', '', clean)
    clean = clean.replace(" г,", " город,")
    clean = clean.replace(" ул,", " улица,")
    clean = clean.replace(" д,", " дом,")
    clean = clean.strip(", ").strip()
    
    url = "https://geocode-maps.yandex.ru/1.x/"
    params = {
        "geocode": clean + ", Россия",
        "format": "json",
        "apikey": API_KEY,
        "results": 1
    }
    
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        geo_objects = data['response']['GeoObjectCollection']['featureMember']
        
        if geo_objects:
            pos = geo_objects[0]['GeoObject']['Point']['pos']
            lon, lat = map(float, pos.split())
            return lat, lon
    except:
        pass
    
    return None, None

df_top500['lat'] = None
df_top500['lon'] = None

success = 0
for i, row in df_top500.iterrows():
    if pd.notna(row['address']):
        lat, lon = get_coords_yandex(row['address'])
        df_top500.at[i, 'lat'] = lat
        df_top500.at[i, 'lon'] = lon
        
        if lat:
            success += 1
        
        if success % 50 == 0 and success > 0:
            print(f"Найдено {success}")
    
    time.sleep(0.2)

print(f"Итого найдено {success} из {len(df_top500)}")

Введите свой API для Яндекс гео:
Найдено 50
Найдено 100
Найдено 150
Найдено 200
Найдено 250
Найдено 300
Найдено 350
Найдено 400
Найдено 450
Итого найдено 498 из 500


**Построение карты**

In [6]:
df_map = df_top500.dropna(subset=['lat', 'lon'])

fig = px.scatter_map(
    df_map,
    lat = "lat",
    lon = "lon",
    hover_name = "company_name",
    hover_data = ['total_revenue', 'address'],
    size = 'total_revenue',
    size_max = 30,
    zoom = 2,
    height = 600,
    center = {"lat": 55.7, "lon": 80.0}
)

fig.update_layout(map_style = "open-street-map")
fig.show()